# Shadow Nano Transformer — Colab Training
**Run this notebook top to bottom. Don't skip any cell.**

Before starting:
- Runtime → Change runtime type → T4 GPU → Save
- Upload `poems_pure_lyric_clean.txt` and `shadow_transformer.py` when prompted

In [ ]:
# CELL 1 — Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# CELL 2 — Upload your files
from google.colab import files

print('Upload poems_pure_lyric_clean.txt and shadow_transformer.py')
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

In [ ]:
# CELL 3 — Verify files are present
import os
from pathlib import Path

corpus = Path('poems_pure_lyric_clean.txt')
script = Path('shadow_transformer.py')

assert corpus.exists(), 'poems_pure_lyric_clean.txt not found!'
assert script.exists(), 'shadow_transformer.py not found!'

text = corpus.read_text(encoding='utf-8')
print(f'Corpus size: {len(text):,} chars')
print(f'First 200 chars preview:')
print(text[:200])

In [ ]:
# CELL 4 — Patch shadow_transformer.py to use GPU
# This replaces the hardcoded CPU device with auto-detection

content = Path('shadow_transformer.py').read_text(encoding='utf-8')

# Replace CPU-only device line
content = content.replace(
    'device = torch.device("cpu")',
    'device = torch.device("cuda" if torch.cuda.is_available() else "cpu")'
)

Path('shadow_transformer.py').write_text(content, encoding='utf-8')
print('Patched shadow_transformer.py to use GPU')

# Verify patch
patched = Path('shadow_transformer.py').read_text()
if 'cuda' in patched:
    print('GPU patch confirmed!')
else:
    print('WARNING: patch may have failed, check manually')

In [ ]:
# CELL 5 — Train from scratch with full settings
# Expected time: 3-4 hours on T4
# DO NOT close the browser tab while this runs
# The output will show epoch-by-epoch progress

!python shadow_transformer.py \
    --mode train \
    --text-path poems_pure_lyric_clean.txt \
    --checkpoint shadow_nano_colab.pt \
    --max-vocab 7000 \
    --min-freq 2 \
    --emb-dim 128 \
    --num-heads 4 \
    --num-layers 4 \
    --ffn-dim 512 \
    --seq-len 128 \
    --dropout 0.1 \
    --epochs 50 \
    --steps-per-epoch 500 \
    --batch-size 64 \
    --lr 3e-4 \
    --grad-clip 1.0 \
    --warmup-steps 500 \
    --patience 10 \
    --val-split 0.03 \
    --seed 42

In [ ]:
# CELL 6 — Verify checkpoint was saved
ckpt = Path('shadow_nano_colab.pt')
assert ckpt.exists(), 'Checkpoint not found! Training may have failed.'
size_mb = ckpt.stat().st_size / 1e6
print(f'Checkpoint saved: {ckpt} ({size_mb:.1f} MB)')

In [ ]:
# CELL 7 — Quick generation test
# Should produce actual poetry, not gibberish

!python shadow_transformer.py \
    --mode generate_poems \
    --checkpoint shadow_nano_colab.pt \
    --prompt "In the quiet night" \
    --prompt "I miss her" \
    --prompt "When she is gone" \
    --poem-count 2 \
    --lines-per-poem 4 \
    --line-retries 20 \
    --poem-candidates 12 \
    --temperature 0.74 \
    --samples-log colab_test_output.txt

print('\n--- OUTPUT ---')
print(Path('colab_test_output.txt').read_text(encoding='utf-8'))

In [ ]:
# CELL 8 — Download everything
import shutil
from google.colab import files

# Bundle checkpoint + test output
shutil.make_archive('shadow_nano_colab_results', 'zip', '.', 'shadow_nano_colab.pt')

# Download individually
files.download('shadow_nano_colab.pt')       # The trained model
files.download('colab_test_output.txt')       # Sample poems
files.download('shadow_transformer.py')       # GPU-patched version